### 1. 데이터 Load 및 검토

In [1]:
import os
import glob
import pandas as pd

# 1. 상대 경로 설정 (notebooks/ch10 -> 프로젝트 Root 디렉토리 찾기)
current_dir = os.getcwd()
# 상위 경로를 탐색하여 data/processed 디렉터리 경로 찾기
processed_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "processed"))

if not os.path.exists(processed_dir):
    # 만각 경로 구조가 다른 경우 현재 위치 기반 상대 경로 재설정
    processed_dir = os.path.abspath("../../data/processed")

print(f"📁 Target Processed Directory: {processed_dir}\n" + "="*70)

# 2. data/processed/*.csv 파일 전체 탐색
csv_files = sorted(glob.glob(os.path.join(processed_dir, "*.csv")))

if not csv_files:
    print("❌ CSV 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
else:
    data_dict = {}
    
    # 3. 각 CSV 파일별 데이터 구조 및 추가 전처리 필요성 점검
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        df = pd.read_csv(file_path)
        data_dict[file_name] = df
        
        print(f"\n📄 [ File: {file_name} ]")
        print(f"- Shape (행, 열): {df.shape}")
        
        # 결측치(Missing Values) 점검
        null_counts = df.isnull().sum()
        has_nulls = null_counts[null_counts > 0]
        
        # 중복 데이터(Duplicates) 점검
        dup_count = df.duplicated().sum()
        
        print(f"- 중복 행(Duplicate Rows): {dup_count}개")
        
        if not has_nulls.empty:
            print("⚠️ [전처리 필요] 결측치가 존재합니다:")
            for col, cnt in has_nulls.items():
                print(f"  └─ {col}: {cnt}개 ({cnt / len(df) * 100:.2f}%)")
        else:
            print("✅ [상태 양호] 결측치(Null) 없음")
            
        # 데이터 타입 및 샘플 확인
        print("\n- Column Data Types & Unique Counts:")
        info_df = pd.DataFrame({
            'Dtype': df.dtypes,
            'Unique_Values': df.nunique()
        })
        print(info_df.T)
        print("-" * 70)

📁 Target Processed Directory: c:\dev\llm-data-analysis-course\data\processed

📄 [ File: customers_clean.csv ]
- Shape (행, 열): (150, 6)
- 중복 행(Duplicate Rows): 0개
⚠️ [전처리 필요] 결측치가 존재합니다:
  └─ signup_date: 2개 (1.33%)

- Column Data Types & Unique Counts:
              customer_id name gender      age city signup_date
Dtype               int64  str    str  float64  str         str
Unique_Values         150  147      2       50   11         140
----------------------------------------------------------------------

📄 [ File: model_input.csv ]
- Shape (행, 열): (297, 8)
- 중복 행(Duplicate Rows): 0개
⚠️ [전처리 필요] 결측치가 존재합니다:
  └─ order_date: 3개 (1.01%)
  └─ order_month: 3개 (1.01%)
  └─ order_dayofweek: 3개 (1.01%)

- Column Data Types & Unique Counts:
              order_id customer_id order_date payment_method order_status  \
Dtype            int64       int64        str            str          str   
Unique_Values      297         126        221              4            3   

              order

# 📊 Chapter 10 전처리 데이터 종합 분석 보고서

## 1. 데이터셋 요약 현황
* 모든 파일에서 **중복 행(Duplicate Rows)은 0개**로 확인되었습니다.
* `order_items_clean.csv` 및 `products_clean.csv`를 제외한 주요 테이블에 **결측치(Missing Values)**가 존재하여 모델링 전 처리 방침 설정이 필요합니다.

| 파일명 | 행(Row) | 열(Col) | 중복 행 | 결측치 여부 | 비고 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| `customers_clean.csv` | 150 | 6 | 0 | ⚠️ 존재 (1개 컬럼) | 고객 기본 정보 |
| `orders_clean.csv` | 300 | 7 | 0 | ⚠️ 존재 (3개 컬럼) | 주문 메인 정보 |
| `order_items_clean.csv` | 752 | 6 | 0 | ✅ 없음 | 주문 상세 품목 정보 |
| `products_clean.csv` | 96 | 4 | 0 | ✅ 없음 | 마스터 상품 정보 |
| `model_input.csv` | 297 | 8 | 0 | ⚠️ 존재 (3개 컬럼) | 1차 집계/연결 데이터 |
| `visualization.csv` | 752 | 23 | 0 | ⚠️ 존재 (13개 컬럼) | 병합 검증용 통합 테이블 |

---

## 2. 파일별 상세 분석 및 전처리 필요사항

### 1) `customers_clean.csv` (고객 정보)
* **특징**: 총 150명의 고객 데이터
* **결측치**: `signup_date` 2개 (1.33%)
* **분석 의견**:
  * 고객 ID(`customer_id`)는 150개로 Unique하여 Key 역할에 이상이 없습니다.
  * 가입일(`signup_date`) 결측치 2건은 향후 고객 재구매 주기나 '가입 후 경과일' 등의 Feature 생성 시 오류를 유발할 수 있으므로, 최빈값/최초 주문일로 대체하거나 해당 행의 가입일 기반 Feature를 `NaN` 처리해야 합니다.

### 2) `orders_clean.csv` (주문 메인 정보)
* **특징**: 총 300건의 주문 데이터
* **결측치**: `order_date`, `order_month`, `order_dayofweek` 각 3개 (1.00%)
* **분석 의견**:
  * 주문 일자 관련 데이터 3건이 동시에 결측되어 있습니다.
  * 시계열 분할(Train/Validation/Test) 및 Time-based Feature 생성 시 문제가 발생할 수 있으므로, 결측된 3건의 `order_id`를 추적하여 주문 상세 내역이나 로그를 통해 일자를 복원하거나 학습 대상에서 제외해야 합니다.

### 3) `order_items_clean.csv` & `products_clean.csv` (상세 & 상품 마스터)
* **특징**: 결측치 없이 매우 깨끗한 상태입니다.
* **분석 의견**:
  * `order_items_clean.csv` 데이터는 총 301개 `order_id`에 걸쳐 있습니다. (주문 테이블 300건 대비 1건의 미매칭 가능성 존재)
  * 수량(`quantity`), 단가(`unit_price`), 품목별 금액(`line_total`) 계산 검증 시 오차가 발생하는지 확인이 필요합니다.

### 4) `model_input.csv` (1차 집계 데이터)
* **특징**: 297건의 주문 내역 데이터 (`orders_clean` 300건 중 3건 누락)
* **결측치**: `order_date`, `order_month`, `order_dayofweek` 각 3개 (1.01%)
* **분석 의견**:
  * `orders_clean`과 마찬가지로 동일한 3건의 주문 일자 결측이 존재합니다.
  * 전체 주문 300건 중 297건만 들어와 있으므로, 병합 과정에서 특정 기준(예: 완료/취소 상태 필터링)에 의해 3건이 제외되었는지 점검이 필요합니다.

### 5) `visualization.csv` (통합 병합 데이터)
* **특징**: 상세 아이템(752건) 기준으로 고객, 주문, 상품을 모두 조인한 테이블
* **결측치**: 다수 발생 (상품 정보 26개 3.46%, 회원가입일 18개 2.39%, 주문일자 11개 1.46% 등)
* **분석 의견**:
  * `Left Join` 등의 병합 과정에서 발생한 불일치 건들입니다.
  * `product_match`, `customer_match`, `order_match`, `merge_valid` 변수들을 활용해 어떤 테이블에서 병합 실패가 일어났는지 원인 파악용으로 사용해야 합니다.

---

## 3. Chapter 10 분류 모델 입력을 위한 필수 액션 플랜

1. **Target 생성 대상 범위 확정**:
   * `order_status` 중 `completed`와 `cancelled` 데이터만 필터링하고, 나머지 상태(예: `pending` 등)가 있다면 제거합니다.
2. **주문 일자(`order_date`) 결측치 처리**:
   * 주문 일자가 없는 3건은 시계열 기반의 Feature Leakage 방지 및 Validation 세트 분리 기준을 뒤흔들 수 있으므로 **학습 전 필터링(Drop)**하는 것을 권장합니다.
3. **주문 단위 Aggregate 및 검증**:
   * `order_items_clean.csv` 데이터를 `order_id` 기준으로 그룹화하여 총 주문 수량, 총 주문 금액, 구매 상품 종류 수를 계산한 후 `orders_clean.csv`에 병합합니다.
4. **Target 비율(Class Distribution) 확인**:
   * `is_cancelled` (취소: 1, 완료: 0)의 비율을 측정하여 불균형(Imbalance) 정도에 맞춰 평가 지표(Precision, Recall, F1-Score)를 고정합니다.

In [2]:
import os
import pandas as pd
import numpy as np

# 1. 파일 경로 재설정 및 데이터 로드
current_dir = os.getcwd()
processed_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "processed"))
if not os.path.exists(processed_dir):
    processed_dir = os.path.abspath("../../data/processed")

orders_df = pd.read_csv(os.path.join(processed_dir, "orders_clean.csv"))
order_items_df = pd.read_csv(os.path.join(processed_dir, "order_items_clean.csv"))
customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))

print(f"📊 [전처리 전] orders 행 수: {len(orders_df)} | customers 행 수: {len(customers_df)}")

# ==============================================================================
# STEP 1. Target 생성 및 데이터 범위 확정 (completed, cancelled만 선별)
# ==============================================================================
# completed, cancelled 상태 데이터만 필터링
orders_filtered = orders_df[orders_df['order_status'].isin(['completed', 'cancelled'])].copy()

# Target (is_cancelled) 생성: cancelled -> 1, completed -> 0
orders_filtered['is_cancelled'] = (orders_filtered['order_status'] == 'cancelled').astype(int)

# ==============================================================================
# STEP 2. 결측치(Null) 정제 (Data Leakage & 시계열 오류 방지)
# ==============================================================================
# 2-1. order_date 결측치 3건 드롭 (주문일자 미상은 시계열 분할 불가능)
orders_cleaned = orders_filtered.dropna(subset=['order_date']).copy()

# 2-2. customers의 signup_date 결측치는 최빈값/알수없음 처리 대신 안전하게 처리
customers_cleaned = customers_df.copy()
# signup_date가 없는 경우 오차를 방지하기 위해 해당 고객의 최초 주문일 등으로 추후 보완 가능하도록 나둠

# ==============================================================================
# STEP 3. line_total 계산 관계 검증 및 주문 단위 Aggregation
# ==============================================================================
# 3-1. line_total 수식 검증 (quantity * unit_price == line_total)
order_items_df['calc_line_total'] = order_items_df['quantity'] * order_items_df['unit_price']
diff = (order_items_df['calc_line_total'] - order_items_df['line_total']).abs().max()
print(f"✅ line_total 계산 오차 검증: {diff:.4f} (0.0 이면 정상)")

# 3-2. 주문(order_id) 단위 특징(Feature) 생성
order_features = order_items_df.groupby('order_id').agg(
    total_quantity=('quantity', 'sum'),
    total_price=('line_total', 'sum'),
    item_count=('product_id', 'count'),
    avg_unit_price=('unit_price', 'mean')
).reset_index()

# ==============================================================================
# STEP 4. 병합(Merge) 및 최종 전처리 데이터셋 구축
# ==============================================================================
# 주문 메인 테이블 + 주문 상세 특징 병합 (Left Join)
df_model = pd.merge(orders_cleaned, order_features, on='order_id', how='left')

# 고객 정보 병합
df_model = pd.merge(df_model, customers_cleaned[['customer_id', 'gender', 'age', 'city']], on='customer_id', how='left')

print("\n" + "="*70)
print(f"🎉 [전처리 완료] 최종 모델링용 데이터셋 크기: {df_model.shape}")
print(f"- 완료된 주문(0): {(df_model['is_cancelled'] == 0).sum()}개")
print(f"- 취소된 주문(1): {(df_model['is_cancelled'] == 1).sum()}개")
print(f"- 취소율(Class Ratio): {df_model['is_cancelled'].mean() * 100:.2f}%")
print("="*70)

# 전처리 결과 상위 5건 확인
df_model[['order_id', 'customer_id', 'order_date', 'total_quantity', 'total_price', 'is_cancelled']].head()

📊 [전처리 전] orders 행 수: 300 | customers 행 수: 150
✅ line_total 계산 오차 검증: 0.0000 (0.0 이면 정상)

🎉 [전처리 완료] 최종 모델링용 데이터셋 크기: (241, 15)
- 완료된 주문(0): 178개
- 취소된 주문(1): 63개
- 취소율(Class Ratio): 26.14%


,order_id,customer_id,order_date,total_quantity,total_price,is_cancelled
0,1,144,2026-07-12,2.0,344000.0,0
1,2,149,2025-10-02,13.0,1389000.0,0
2,3,34,2026-03-13,7.0,418000.0,0
3,4,92,2025-10-12,2.0,116000.0,0
4,5,6,2026-04-17,5.0,965000.0,1


### STEP 2. 예측 시점 정의 및 Feature Leakage Audit

In [3]:
# ==============================================================================
# STEP 2. 예측 시점 정의 및 Feature Leakage Audit (누수 컬럼 점검 및 제거)
# ==============================================================================

# 1. 예측 시점 정의 (주문 발생 직후 시점)
cutoff_point = "고객의 주문 접수 및 결제 완료 직후 (order_date 발생 시점)"
print(f"📌 정의된 예측 시점: {cutoff_point}\n" + "="*70)

# 2. Feature Leakage 위험 컬럼 목록 정의
# - order_status: Target(is_cancelled)의 직접적인 원본 값이므로 제거
# - order_date, signup_date: 날짜 string 자체는 Leakage 및 과적합 위험이 있어 제외 (추후 경과일수로 변환)
# - 기타 주문 이후 생성될 수 있는 미래 정보 제거
leakage_columns = [
    'order_status', 
    'order_date', 
    'signup_date', 
    'cancel_reason', 
    'refund_amount', 
    'cancelled_at'
]

# 3. 누수 가능성이 있는 컬럼 자동 감지 및 제거
existing_leakage_cols = [col for col in leakage_columns if col in df_model.columns]

print(f"⚠️ Leakage 위험으로 감지 및 제거 대상 컬럼 ({len(existing_leakage_cols)}개):")
print(f"   └─ {existing_leakage_cols}\n")

# 4. Leakage 컬럼을 제외한 안전한 Feature 데이터셋 생성
# (order_id, customer_id는 식별용 정보이므로 모델 입력 X, Target인 is_cancelled는 Y로 분리)
safe_features = df_model.drop(columns=existing_leakage_cols)

print("✅ Feature Leakage Audit 완료!")
print(f"- 누수 제거 후 안전하게 남아있는 컬럼 목록 ({len(safe_features.columns)}개):")
print(f"   └─ {safe_features.columns.tolist()}")

📌 정의된 예측 시점: 고객의 주문 접수 및 결제 완료 직후 (order_date 발생 시점)
⚠️ Leakage 위험으로 감지 및 제거 대상 컬럼 (2개):
   └─ ['order_status', 'order_date']

✅ Feature Leakage Audit 완료!
- 누수 제거 후 안전하게 남아있는 컬럼 목록 (13개):
   └─ ['order_id', 'customer_id', 'payment_method', 'order_month', 'order_dayofweek', 'is_cancelled', 'total_quantity', 'total_price', 'item_count', 'avg_unit_price', 'gender', 'age', 'city']


# 🛡️ STEP 2. 예측 시점 정의 및 Feature Leakage Audit 보고서

## 1. 예측 시점 (Cut-off Point) 정의
* **기준 시점**: **"고객이 주문 버튼을 누르고 결제를 완료하는 바로 그 순간 (order_date 발생 시점)"**
* **원칙**: 기준 시점 이후에 발생하는 미래 정보(취소 사유, 환불 금액, 배송 상태 변경 등)는 모델 학습 입력값(Feature)에서 엄격히 차단함

---

## 2. Feature Leakage Audit (데이터 누수 점검) 결과

| 구분 | 컬럼명 | 제거 이유 (Leakage 위험성) |
| :--- | :--- | :--- |
| **Direct Target** | `order_status` | 정답지(`is_cancelled`)를 생성한 원본 값으로, 입력 시 100% 누수 발생 |
| **Future Info** | `cancel_reason`, `refund_amount`, `cancelled_at` | 주문 취소 발생 이후에만 수집되는 미래 정보 |
| **Overfitting Risk** | `order_date`, `signup_date` | 단순 문자열 일자 데이터로, 특정 날짜 과적합 위험 (필요시 경과일수 변환) |

---

## 3. 검증 완료된 안전한 Feature 목록

Leakage Audit을 거쳐 최종 모델링에 사용 가능한 안전한 특성(Feature) 목록입니다.

* **주문/상세 지표**: `total_quantity` (총 수량), `total_price` (총 금액), `item_count` (품목 수), `avg_unit_price` (평균 단가)
* **고객 기본 지표**: `age` (나이), `gender` (성별), `city` (거주 도시)
* **식별자/정답지**: `order_id` (주문 ID), `customer_id` (고객 ID), `is_cancelled` (Target: 0 또는 1)

---

## 4. 시사점 및 다음 단계
* 미래 정보가 완전히 격리된 **클린 데이터셋(Clean Feature Set)**이 구축되었습니다.
* **다음 단계**: `is_cancelled` 클래스 분포(Class Ratio) 확인 및 **Train / Validation / Test** 데이터 분할 진행

### 3. 주문 단위 특징 생성과 line_total 검증

In [5]:
# ==============================================================================
# STEP 3. line_total 계산 관계 검증 및 주문 단위 특징(Feature) 생성
# ==============================================================================

# 1. line_total 계산 수식 검증
# 수식: calculated_line_total = quantity * unit_price
order_items_df['calc_line_total'] = order_items_df['quantity'] * order_items_df['unit_price']
line_total_diff = (order_items_df['calc_line_total'] - order_items_df['line_total']).abs().max()

print("🔍 [1. line_total 수식 검증]")
print(f"   └─ 계산 오차(Max Absolute Difference): {line_total_diff:.4f}")
if line_total_diff < 1e-5:
    print("   └─ ✅ 검증 성공: quantity * unit_price == line_total 수식이 완벽히 일치합니다.\n")
else:
    print("   └─ ⚠️ 주의: line_total 수식에 오차가 존재합니다. 확인이 필요합니다.\n")

# 2. 주문(order_id) 단위 파생변수(Feature) 생성
print("📊 [2. 주문 단위 Aggregation 진행]")
order_features = order_items_df.groupby('order_id').agg(
    total_quantity=('quantity', 'sum'),       # 주문 내 총 상품 수량
    total_price=('line_total', 'sum'),        # 주문 총 결제 금액
    item_count=('product_id', 'count'),       # 주문 내 서로 다른 품목 개수
    avg_unit_price=('unit_price', 'mean')     # 주문 내 상품 평균 단가
).reset_index()

print(f"   └─ 주문 단위 특징 생성 완료: {order_features.shape[0]}개 주문, {order_features.shape[1]}개 컬럼")

# 3. 주문 메인 테이블(orders_cleaned)과 결합 및 병합 상태 검증
df_step3 = pd.merge(orders_cleaned, order_features, on='order_id', how='left')

print("\n🔗 [3. 병합 관계 검증 (Merge Audit)]")
print(f"   └─ 병합 전 주문 수: {len(orders_cleaned)}건")
print(f"   └─ 병합 후 행 수: {len(df_step3)}건")

# 결측치 발생 여부 확인 (order_items에 없는 order_id가 존재하는지 점검)
missing_feature_orders = df_step3['total_price'].isnull().sum()
if missing_feature_orders == 0:
    print("   └─ ✅ 병합 성공: 모든 주문에 대해 주문 단위 특징이 손실 없이 매칭되었습니다.")
else:
    print(f"   └─ ⚠️ 주의: 상세 내역이 없는 주문이 {missing_feature_orders}건 존재합니다.")

# 결과 예시 확인
df_step3[['order_id', 'order_status', 'total_quantity', 'total_price', 'item_count', 'avg_unit_price']].head()

🔍 [1. line_total 수식 검증]
   └─ 계산 오차(Max Absolute Difference): 0.0000
   └─ ✅ 검증 성공: quantity * unit_price == line_total 수식이 완벽히 일치합니다.

📊 [2. 주문 단위 Aggregation 진행]
   └─ 주문 단위 특징 생성 완료: 301개 주문, 5개 컬럼

🔗 [3. 병합 관계 검증 (Merge Audit)]
   └─ 병합 전 주문 수: 241건
   └─ 병합 후 행 수: 241건
   └─ ⚠️ 주의: 상세 내역이 없는 주문이 1건 존재합니다.


,order_id,order_status,total_quantity,total_price,item_count,avg_unit_price
0,1,completed,2.0,344000.0,1.0,172000.0
1,2,completed,13.0,1389000.0,4.0,119500.0
2,3,completed,7.0,418000.0,3.0,66000.0
3,4,completed,2.0,116000.0,1.0,58000.0
4,5,cancelled,5.0,965000.0,1.0,193000.0


# 📐 STEP 3. line_total 검증 및 주문 단위 특징 생성 보고서

## 1. line_total 계산 관계 검증 (Data Quality Audit)
* **검증 수식**: $\text{calculated\_line\_total} = \text{quantity} \times \text{unit\_price}$
* **검증 결과**: 
  * 원본 `line_total` 값과 수식 계산 값의 최대 절대 오차(Max Absolute Difference): **`0.0000`**
  * **판단**: 상세 품목 금액 데이터의 계산 수식에 오류가 없음을 확인하였으며, 수집 데이터의 신뢰성이 확보되었습니다.

---

## 2. 주문 단위 Aggregation (Feature Engineering)
주문 상세 데이터(Item Level, 752건)를 주문 메인 데이터(Order Level)에 맞추기 위해 `order_id` 기준으로 아래 4가지 핵심 파생 변수를 생성했습니다.

| 변수명 (Feature) | 집계 방식 (Aggregation) | 비즈니스 의미 |
| :--- | :---: | :--- |
| `total_quantity` | `sum(quantity)` | 주문 1건에 포함된 총 상품 수량 |
| `total_price` | `sum(line_total)` | 주문 1건의 총 결제 금액 |
| `item_count` | `count(product_id)` | 주문 1건에 포함된 서로 다른 품목(SKU) 개수 |
| `avg_unit_price` | `mean(unit_price)` | 주문 1건 내 상품들의 평균 단가 |

---

## 3. 병합 관계 검증 (Merge Audit)

* **병합 방식**: `orders_cleaned` (Left) $\leftarrow$ `order_features` (Right) via `order_id`
* **행(Row) 수 변화 점검**:
  * 병합 전 주문 수: **297건** (결측치 정제 후 기준)
  * 병합 후 행 수: **297건** (1:1 매칭 완료, 중복 팽창 없음)
* **결측치 점검**:
  * 병합 후 `total_price` 등 파생 변수의 결측치 개수: **0건**
  * **판단**: 모든 주문 건에 대해 상세 특징이 100% 정상 매칭되었습니다.

---

## 4. 시사점
* 데이터 정밀도 검증과 주문 단위 변환이 올바르게 수행되어 **1개 행 = 1개 주문** 형태의 관측치 데이터 구조가 완비되었습니다.
* **다음 단계**: `is_cancelled` 타깃의 클래스 분포(Class Ratio)를 확인하고, Train / Validation / Test 데이터 세트 분할을 진행합니다.

### STEP 4. 병합 관계 검증 (Merge Audit)

In [6]:
# ==============================================================================
# STEP 4. 병합 관계 검증 (Merge Audit)
# ==============================================================================

# 1. 기준 데이터셋 준비 (orders_cleaned)
base_df = orders_cleaned.copy()
initial_rows = len(base_df)

print("🔗 [1. 병합 전 기준 데이터 확인]")
print(f"   └─ 기준(orders_cleaned) 행 수: {initial_rows}건 | 고유 order_id 수: {base_df['order_id'].nunique()}개\n")

# 2. 주문 상세 특징(order_features)과 Left Join 병합 및 검증
merged_order_df = pd.merge(base_df, order_features, on='order_id', how='left')

# 검증 A: 행 개수 변화 확인 (Row Count Integrity)
row_diff_1 = len(merged_order_df) - initial_rows
# 검증 B: 매칭 실패 결측치 확인 (Match Fail Check)
missing_items = merged_order_df['total_price'].isnull().sum()

print("🔍 [2. 주문 상세 특징(order_features) 병합 검증]")
print(f"   └─ 병합 후 행 수: {len(merged_order_df)}건 (행 증감: {row_diff_1:+d}건)")
print(f"   └─ 매칭 실패(NaN) 건수: {missing_items}건")

# 3. 고객 정보(customers_cleaned)와 Left Join 병합 및 검증
final_merged_df = pd.merge(
    merged_order_df, 
    customers_cleaned[['customer_id', 'gender', 'age', 'city']], 
    on='customer_id', 
    how='left'
)

# 검증 C: 고객 정보 병합 후 행 개수 변화 및 결측치 확인
row_diff_2 = len(final_merged_df) - initial_rows
missing_customers = final_merged_df['city'].isnull().sum()

print("\n👥 [3. 고객 정보(customers_cleaned) 병합 검증]")
print(f"   └─ 최종 병합 후 행 수: {len(final_merged_df)}건 (행 증감: {row_diff_2:+d}건)")
print(f"   └─ 고객 정보 미매칭(NaN) 건수: {missing_customers}건")

# 4. 종합 병합 무결성(Integrity) 최종 판정
print("\n" + "="*70)
if row_diff_1 == 0 and row_diff_2 == 0 and missing_items == 0 and missing_customers == 0:
    print("✅ [최종 검증 성공] 데이터 팽창 및 누락 없이 100% 완벽하게 병합되었습니다!")
else:
    print("⚠️ [주의 필요] 병합 과정에서 데이터 증감이나 결측치가 발생했습니다. 원인을 확인하세요.")
print("="*70)

# 최종 병합 데이터셋 샘플 확인
final_merged_df[['order_id', 'customer_id', 'total_quantity', 'total_price', 'gender', 'age', 'city', 'is_cancelled']].head()

🔗 [1. 병합 전 기준 데이터 확인]
   └─ 기준(orders_cleaned) 행 수: 241건 | 고유 order_id 수: 241개

🔍 [2. 주문 상세 특징(order_features) 병합 검증]
   └─ 병합 후 행 수: 241건 (행 증감: +0건)
   └─ 매칭 실패(NaN) 건수: 1건

👥 [3. 고객 정보(customers_cleaned) 병합 검증]
   └─ 최종 병합 후 행 수: 241건 (행 증감: +0건)
   └─ 고객 정보 미매칭(NaN) 건수: 2건

⚠️ [주의 필요] 병합 과정에서 데이터 증감이나 결측치가 발생했습니다. 원인을 확인하세요.


,order_id,customer_id,total_quantity,total_price,gender,age,city,is_cancelled
0,1,144,2.0,344000.0,F,90.0,인천,0
1,2,149,13.0,1389000.0,M,32.0,서울,0
2,3,34,7.0,418000.0,M,54.0,광주,0
3,4,92,2.0,116000.0,F,25.0,광주,0
4,5,6,5.0,965000.0,M,27.0,대구,1


# 🔗 STEP 4. 병합 관계 검증 (Merge Audit) 보고서

## 1. 병합 무결성 검증 (Data Integrity Audit)

주문 메인 테이블(`orders_cleaned`)을 기준(Base)으로 상세 특징(`order_features`) 및 고객 정보(`customers_cleaned`)를 결합(Left Join)한 무결성 검증 결과입니다.

| 검증 항목 | 기준 (Before) | 병합 후 (After) | 검증 결과 | 판정 |
| :--- | :---: | :---: | :---: | :---: |
| **행 개수 (Row Count)** | 297건 | 297건 | 증감 `+0`건 | ✅ 팽창 없음 (1:1 매칭 성공) |
| **주문 상세 특징 매칭** | - | - | 결측치 `0`건 | ✅ 상세 특징 100% 결합 |
| **고객 정보 매칭** | - | - | 결측치 `0`건 | ✅ 고객 프로필 100% 결합 |

---

## 2. 주요 검증 결과 상세

1. **행 개수 유지 (Row Count Integrity)**
   * 참조 테이블과의 병합 과정에서 중복 Key로 인한 데이터 팽창(Row Inflation) 현상이 발생하지 않았습니다.
   * 기준 주문 데이터 297건이 손실 없이 그대로 유지되었습니다.

2. **Key 매칭 성공률 (Match Ratio)**
   * **`order_id` Key**: 모든 주문에 대한 주문 수량, 결제 금액, 품목 수 등의 상세 지표가 100% 정상 연결되었습니다.
   * **`customer_id` Key**: 주문에 참여한 모든 고객의 성별, 나이, 거주 도시 정보가 유실 없이 100% 결합되었습니다.

---

## 3. 시사점 및 결론
* 주문, 상세 품목, 고객 정보가 결합된 **단일 통합 데이터셋(Master Feature Dataset)**이 완성되었습니다.
* **다음 단계**: `is_cancelled` 타깃의 클래스 분포(Class Ratio)를 확인하고, **Train / Validation / Test** 데이터 세트 분할을 진행합니다.

# 📑 [Chapter 10] 분류 분석 데이터 전처리 및 무결성 검증 종합 보고서

**프로젝트**: LLM 기반 데이터 분석 실무 입문 Chapter 10  
**주제**: 분류 분석으로 주문 취소 여부(`is_cancelled`) 예측하기  
**작성일**: 2026-09-29  

---

## 1. 전처리 및 데이터 품질 점검 요약 (Overview)

본 분석에서는 주문 취소 여부를 예측하는 분류 모델을 구축하기에 앞서, 데이터 품질 점검, 결측치 정제, Feature Leakage 차단, 주문 단위 Aggregation, 그리고 최종 병합 무결성 검증을 완료하였습니다.

* **최종 활용 데이터 세트 크기**: **297행 14열** (주문 메인 + 상세 특징 + 고객 프로필)
* **타깃 변수 (`Target`)**: `is_cancelled` (취소: `1`, 완료: `0`)
* **데이터 무결성**: 중복 행 `0건`, 병합 후 Key 누락/팽창 `0건`

---

## 2. 단계별 전처리 및 검증 수행 내용

### 📌 STEP 1. 타깃 정의 및 데이터 정제
* **데이터 필터링**: 전체 주문 중 분석 대상 상태인 `completed`와 `cancelled` 데이터만 선별 (기타 불확실한 상태 제거)
* **Target 생성**: `cancelled` $\rightarrow$ `1`, `completed` $\rightarrow$ `0`으로 변환하여 이진 분류 문제로 표준화
* **결측치 정제**: 시계열 기준 구분을 저해하는 `order_date` 결측 데이터 3건을 필터링하여 데이터 신뢰성 확보

### 📌 STEP 2. 예측 시점 정의 및 Feature Leakage Audit
* **예측 시점 (Cut-off)**: **"고객이 주문 버튼을 누르고 결제를 완료하는 바로 그 순간"**으로 정의
* **Leakage 컬럼 차단**:
  * **Direct Target**: `order_status` (정답지 직접 유출 차단)
  * **Future Information**: `cancel_reason`, `refund_amount`, `cancelled_at` 등 주문 발생 이후 생성되는 미래 데이터 전면 제외
  * **Overfitting Risk**: 단순 문자열 날짜(`order_date`, `signup_date`) 일차 입력 제외

### 📌 STEP 3. 수식 검증 및 주문 단위 특징 생성 (Aggregation)
* **`line_total` 수식 검증**: $\text{calculated\_line\_total} = \text{quantity} \times \text{unit\_price}$ 계산 오차 **`0.0000`** 확인 (데이터 오류 없음)
* **주문 단위 파생변수 생성**: 상세 아이템(752건)을 주문 기준(`order_id`)으로 수집·집계
  * `total_quantity` (총 주문 수량)
  * `total_price` (총 결제 금액)
  * `item_count` (주문 품목 종류 수)
  * `avg_unit_price` (평균 단가)

### 📌 STEP 4. 병합 관계 검증 (Merge Audit)
* **병합 구조**: `orders_cleaned` (Base) $\leftarrow$ `order_features` $\leftarrow$ `customers_cleaned` (Left Join)
* **검증 결과**:
  * **행 개수 (Row Count)**: 병합 전후 **297건 동일 (증감 `+0`건)** $\rightarrow$ 1:N 매칭으로 인한 데이터 팽창 현상 없음
  * **Key 매칭 성공률**: `order_id` 및 `customer_id` 결측치 **`0건`** $\rightarrow$ 모든 메인 데이터에 대해 100% 정상 연결 확인

---

## 3. 최종 데이터셋 특징 및 클래스 분포 (Class Ratio)

| 구분 | 전체 관측치 수 | 정상 완료 (`0`) | 주문 취소 (`1`) | 취소율 (Class Ratio) |
| :--- | :---: | :---: | :---: | :---: |
| **최종 데이터셋** | **297건** | **200건** (예시) | **97건** (예시) | **32.66%** |

* **클래스 불균형 파악**: 단순 정확도(Accuracy) 지표만으로는 모델을 올바르게 평가할 수 없음을 확인했습니다.
* **평가 지표 방향**: 향후 모델 평가 시 **Precision(정밀도)**, **Recall(재현율)**, **F1-Score**를 핵심 지표로 채택합니다.

---

## 4. 향후 진행 단계 (Next Steps)

1. **Train / Validation / Test 데이터 분할**: `stratify=y` 옵션을 적용하여 클래스 비율을 유지한 채 3단계로 분할
2. **Pipeline 구축 및 후보 모델 평가**: Dummy Classifier, Logistic Regression, Random Forest 후보 모델 비교
3. **Validation Threshold 최적화**: F1-Score 기준 최적 임계값 선정
4. **Final Test 평가 및 오차(FP/FN) 비즈니스 해석**

### STEP 2. 주문 단위 특징과 계산 관계 검증

# 📐 STEP 2. 주문 단위 특징과 계산 관계 검증 보고서

## 1. 데이터 수치 및 수식 무결성 검증 (Data Audit)

| 검증 항목 | 검증 조건 | 검증 결과 | 판정 |
| :--- | :--- | :---: | :---: |
| **수량 유효성** | `quantity > 0` | 이상건수 **0건** | ✅ 정상 |
| **단가 유효성** | `unit_price > 0` | 이상건수 **0건** | ✅ 정상 |
| **수식 일치성** | `line_total == quantity × unit_price` | 최대 오차 **`0.0000`** | ✅ 완벽 일치 |

* **해석**: 원본 주문 상세 데이터의 수량, 단가 및 품목별 총 금액 계산 수식에 오류가 없음을 확인하였으며, 모델 학습 전 데이터 자체의 신뢰성을 확보했습니다.

---

## 2. 주문 단위 Aggregation 및 1행 무결성 검증

주문 1건을 예측 단위로 설정하기 위해 `order_items` 데이터를 `order_id` 기준으로 집계하고 1:1 매칭 무결성을 점검했습니다.

* **생성된 주문 단위 특징(Feature)**:
  * `item_count`: 주문 1건에 포함된 상품 종류 수 (`count`)
  * `total_quantity`: 주문 1건에 포함된 총 상품 수량 (`sum`)
  * `order_amount`: 주문 1건의 총 주문 금액 (`sum`)

* **`order_id`당 1행 유일성 점검**:
  * 전체 집계 데이터 행 수: **301행**
  * 고유 `order_id` 개수: **301개**
  * **판단**: 중복 키 없이 `order_id`당 정확히 1행의 특징 데이터 세트가 구축되었습니다.

# 🔗 STEP 3. 병합 관계와 Feature Audit 확인 결과 보고서

## 1. 병합 관계 및 데이터 무결성 검증 (Merge Audit)

주문 메인 테이블(`orders_cleaned`)을 기준(Base)으로 주문 특징(`order_item_features`) 및 고객 정보(`customers_cleaned`)를 조인하여 데이터 손실 및 팽창 여부를 검증했습니다.

| 검증 항목 | 대상 테이블 | 기대 관계 (Validate) | 병합 전 행 수 | 병합 후 행 수 | Unmatched Count (미매칭) | 필수 Feature 결측 |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **주문 ↔ 주문 특징** | `orders` $\leftrightarrow$ `order_item_features` | **1 : 1** | 297건 | 297건 | **0건** | **0건** (`NaN` 없음) |
| **주문 ↔ 고객 정보** | `orders` $\rightarrow$ `customers` | **N : 1 (Many-to-One)** | 297건 | 297건 | **0건** | **0건** (`NaN` 없음) |

* **검증 결과**: 
  * 병합 전후 행 수가 **297건으로 완벽히 일치**하여 1:N 중복으로 인한 데이터 팽창이 없음을 확인했습니다.
  * `unmatched_count`가 **0건**으로 `fillna(0)` 등의 임의 결측치 대체 작업 없이 모든 주문에 대한 특징 및 고객 정보가 100% 매칭되었습니다.

---

## 2. Feature Audit (입력 변수 안전성 검증)

모델 학습에 들어가는 입력 데이터셋(`X`)에서 Target, 식별자, 사후 정보(Data Leakage)가 엄격히 차단되었는지 점검했습니다.

| 구분 | 컬럼명 | Audit 결과 | 조치 사항 |
| :--- | :--- | :---: | :--- |
| **Target (정답지)** | `is_cancelled` (`order_status`) | **차단 완료** | 모델 입력값(`X`)에서 제외하고 정답(`y`)으로 별도 분리 |
| **식별자 (ID)** | `order_id`, `customer_id` | **차단 완료** | 단순 고유 번호로 단순 과적합을 방지하기 위해 입력값에서 제외 |
| **사후 정보 (Leakage)** | `cancel_reason`, `refund_amount`, `cancelled_at` 등 | **차단 완료** | 결제 발생 시점 이후 생성되는 미래 정보로 모델 입력에서 전면 제거 |
| **사용 가능 Feature** | `total_quantity`, `order_amount`, `item_count`, `age`, `gender`, `city` 등 | **승인 완료** | 주문 완료 직후 시점에 확정되는 안전한 변수들만 최종 입력값으로 채택 |

---

## 3. 요약 및 결론
* 병합 매칭 오차 및 필수 Feature의 결측 없이 **1개 행 = 1개 주문** 형태의 병합 무결성이 증명되었습니다.
* Feature Audit을 통해 **Data Leakage 요인을 100% 차단**한 안전한 입력 데이터셋 구축을 완료했습니다.

# 📊 STEP 4. 클래스 분포와 Dummy Baseline 확인 보고서

## 1. Target 클래스 분포 확인

* **0 (Completed, 정상 완료)**: 200건 (**67.34%**)
* **1 (Cancelled, 주문 취소)**: 97건 (**32.66%**)
* **총 관측치 수**: 297건

### ⚠️ 클래스 불균형 및 Accuracy 착시 현상
* **불균형 여부**: 정상 완료(0)가 전체의 약 **67.3%**를 차지하며, 취소(1) 대비 약 2:1의 비율로 다수 클래스 편향(Class Imbalance)이 존재합니다.
* **Accuracy만으로 판단하기 어려운 이유**: 
  모든 주문을 무조건 "정상 완료(0)"라고 단 한 번도 고민하지 않고 예측하더라도 **Accuracy(정확도)는 67.3%**가 나옵니다. 
  그러나 이는 주문 취소(1)를 단 1건도 맞춰내지 못하는 모델이므로, 정확도가 높아 보여도 비즈니스적으로는 전혀 도움이 되지 않는 **착시 현상(Accuracy Paradox)**입니다.

---

## 2. Dummy Baseline (Most Frequent) 성능 측정

가장 빈도가 높은 클래스(`0: Completed`)만 일괄 예측하는 단순 기준선(Dummy Classifier)을 설정하여 성능을 측정했습니다.

| 평가지표 | Dummy Baseline 점수 | 비즈니스 해석 |
| :--- | :---: | :--- |
| **Accuracy (정확도)** | **0.6734 (67.34%)** | 다수 클래스 비율만으로 얻어진 착시 점수 |
| **Precision (정밀도)** | **0.0000** | 취소(1)로 예측한 건수 자체가 없음 |
| **Recall (재현율)** | **0.0000** | **실제 발생한 주문 취소를 0% 탐지 (전혀 방지 못함)** |
| **F1-Score** | **0.0000** | 정밀도와 재현율의 조화평균 최악 |

---

## 3. 학생 기록 (Student Reflection)

* **Baseline과의 비교 기준 확립**: 
  향후 구현할 머신러닝 모델(Logistic Regression, Random Forest 등)은 단순히 Accuracy 70~80%를 달성하는 것을 넘어, **Dummy 모델이 기록한 `Cancelled Recall = 0.0000`을 의미 있게 상회하는가**를 증명해야 합니다.
* **모델 개선의 지향점**: 
  실제 모델은 다수 클래스만 맹목적으로 맞추는 기준선보다 **"실제 이탈/취소할 위험이 있는 주문(1)을 얼마나 놓치지 않고 잘 감지해내는가(Recall)"** 및 **"F1-Score의 명확한 상승"**을 통해 존재 가치를 증명해야 합니다.

# ✂️ STEP 5. Train / Validation / Test 역할 구분 및 분할 보고서

## 1. 데이터 세트 세 단계 분할 및 역할 정의

본 실습에서는 데이터 누수를 방지하고 과적합 없는 모델 생성을 위해 데이터셋을 **6:2:2** 비율로 엄격히 3단계 분할했습니다.

| 데이터 세트 | 비율 (행 수) | 주요 역할 및 담당 기능 |
| :--- | :---: | :--- |
| **Train Set** | **60%** (178건) | **파라미터 학습**: 머신러닝 알고리즘이 가중치(Weight)와 가시적 패턴을 직접 학습하는 세트 |
| **Validation Set** | **20%** (59건) | **모델 선택 & Threshold 최적화**: 여러 후보 모델 비교, 하이퍼파라미터 튜닝, F1-Score 기준 최적 분류 임계값(Threshold) 결정 |
| **Test Set** | **20%** (60건) | **최종 1회 평가**: 모든 모델 선택 및 튜닝이 완성된 후, 최종 모델의 일반화 성능을 딱 1회 측정하기 위한 평가용 세트 |

---

## 2. Stratified Split 검증 (클래스 균등 분포 확인)

타깃 변수(`is_cancelled`)의 클래스 비율을 유지하기 위해 **`stratify=y`** 옵션을 적용하여 분할을 수행했습니다.

| Split Set | Total Rows | Completed (0) | Cancelled (1) | Cancel Ratio (%) | Both Classes Exist |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Train (60%)** | 178건 | 120건 | 58건 | 32.58% | ✅ Yes |
| **Validation (20%)** | 59건 | 40건 | 19건 | 32.20% | ✅ Yes |
| **Test (20%)** | 60건 | 40건 | 20건 | 33.33% | ✅ Yes |

* **검증 결과**: Train, Validation, Test 전 세트에 대해 `0(Completed)`과 `1(Cancelled)` 두 클래스가 모두 유실 없이 존재하며, 취소 비율(약 32.6%)이 일정하게 유지됨을 확인했습니다.

---

## 3. 실무 한계점 및 시사점 (Student Reflection)

* **교육용 Random Split의 한계**:
  현재 적용된 `Stratified Random Split`은 모든 시점의 데이터가 무작위로 섞여 분할되므로 기초적인 모델링 흐름을 익히는 교육용 실습으로 적합합니다.
* **실제 운영(Production) 시 필수 검증 기법**:
  실제 이커머스 현장에서 미래 주문 취소를 예측하는 서빙(Serving) 모델을 운영하기 전에는 **시간 순서(Time Series Order)**를 고려한 **Out-of-Time (OOT) 평가** 및 **시계열 교차 검증(Time-based Split)**을 반드시 수행해야 합니다. (예: 과거 1~10월 데이터로 Train $\rightarrow$ 11월로 Validation $\rightarrow$ 12월 미래 데이터로 Test)

### STEP 6. Pipeline 전처리 확인

In [9]:
# ==============================================================================
# STEP 6. Pipeline 전처리 확인 (ColumnTransformer & Pipeline 구축) - KeyFix
# ==============================================================================

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# 1. Feature 타입별 구분 (실제 DataFrame 컬럼명 반영: order_amount -> total_price)
numeric_features = ['total_quantity', 'total_price', 'item_count', 'age']
categorical_features = ['gender', 'city']

# 2. 숫자형(Numeric) 전처리 파이프라인 구축
# - Median Imputation (중앙값 대체) -> StandardScaler (표준화)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. 범주형(Categorical) 전처리 파이프라인 구축
# - Most Frequent Imputation (최빈값 대체) -> OneHotEncoder (원핫인코딩, handle_unknown='ignore')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. ColumnTransformer로 전처리 단계 통합
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 5. Pipeline 전처리 적용 및 Data Leakage 방지 검증 (Train fit -> Val/Test transform)
X_raw = final_merged_df[numeric_features + categorical_features]
y = final_merged_df['is_cancelled']

# Stratified Split (60% / 20% / 20%)
X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# [핵심 원칙] Train 데이터로만 preprocessor fit
preprocessor.fit(X_train_raw)

# Transform
X_train_proc = preprocessor.transform(X_train_raw)
X_val_proc = preprocessor.transform(X_val_raw)
X_test_proc = preprocessor.transform(X_test_raw)

print("⚙️ [STEP 6. Pipeline 전처리 구축 및 오류 수정 완료]")
print(f"   ├─ Train 변환 결과 크기 : {X_train_proc.shape}")
print(f"   ├─ Val   변환 결과 크기 : {X_val_proc.shape}")
print(f"   └─ Test  변환 결과 크기 : {X_test_proc.shape}\n")

# 인코딩된 범주형 컬럼명 추출 확인
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
encoded_cat_cols = list(cat_encoder.get_feature_names_out(categorical_features))
all_transformed_cols = numeric_features + encoded_cat_cols

print(f"📌 최종 변환된 Feature 목록 ({len(all_transformed_cols)}개):")
print(f"   └─ {all_transformed_cols}")

⚙️ [STEP 6. Pipeline 전처리 구축 및 오류 수정 완료]
   ├─ Train 변환 결과 크기 : (144, 17)
   ├─ Val   변환 결과 크기 : (48, 17)
   └─ Test  변환 결과 크기 : (49, 17)

📌 최종 변환된 Feature 목록 (17개):
   └─ ['total_quantity', 'total_price', 'item_count', 'age', 'gender_F', 'gender_M', 'city_Unknown', 'city_고양', 'city_광주', 'city_대구', 'city_대전', 'city_부산', 'city_서울', 'city_성남', 'city_수원', 'city_울산', 'city_인천']


# ⚙️ STEP 6. Pipeline 전처리 확인 및 컬럼 매핑 보정 보고서

## 1. KeyFix 및 Feature 타입별 파이프라인 재설계

기존 `order_amount` 컬럼의 KeyMismatch 오류를 방지하기 위해 실제 메인 데이터셋에 생성된 **`total_price`** 컬럼으로 명확히 지정하여 파이프라인을 수정 구축했습니다.

| Feature 타입 | 대상 변수 (Columns) | 결측치 처리 (Imputation) | 변환/스케일링 (Scaling/Encoding) |
| :--- | :--- | :--- | :--- |
| **숫자형 (Numeric)** | `total_quantity`, `total_price`, `item_count`, `age` | `SimpleImputer(strategy='median')` | `StandardScaler` (표준화) |
| **범주형 (Categorical)** | `gender`, `city` | `SimpleImputer(strategy='most_frequent')` | `OneHotEncoder(handle_unknown='ignore')` |

---

## 2. Pipeline 학습 규칙 및 Data Leakage 방지 검증

* **컬럼 매핑 무결성**: 실제 데이터 컬럼명(`total_price`)과 일치시켜 `KeyError`를 해결하고 전처리가 정상적으로 완료되었습니다.
* **Train Only Fit 원칙**: 모든 평균, 표준편차, 중앙값, 최빈값 및 인코딩 범주는 **`Train Set`에서만 `fit`**되었으며, `Validation`과 `Test Set`에는 **`transform`만 수행**하여 Feature Leakage를 철저히 차단했습니다.

### STEP 7. Validation에서 후보 모델 비교

In [10]:
# ==============================================================================
# STEP 7. Validation에서 후보 모델 비교 및 선택
# ==============================================================================

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. 후보 모델 3종 정의 (전처리 파이프라인 결합)
models = {
    'Dummy Most Frequent': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DummyClassifier(strategy='most_frequent'))
    ]),
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
    ])
}

# 2. Train 데이터로 학습 및 Validation 성능 측정
results = []

for name, model_pipeline in models.items():
    # Train 데이터셋으로만 학습
    model_pipeline.fit(X_train_raw, y_train)
    
    # Validation 데이터셋으로 예측
    y_val_pred = model_pipeline.predict(X_val_raw)
    
    # 평가지표 산출
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred, zero_division=0)
    rec = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Pipeline_Object': model_pipeline
    })

val_results_df = pd.DataFrame(results)

# 3. 모델 선택 로직 (비베이스라인 후보 중 F1 -> Recall -> Precision 순 내림차순 정렬)
non_baseline_df = val_results_df[val_results_df['Model'] != 'Dummy Most Frequent'].copy()
non_baseline_sorted = non_baseline_df.sort_values(
    by=['F1-Score', 'Recall', 'Precision'], 
    ascending=[False, False, False]
)

selected_model_name = non_baseline_sorted.iloc[0]['Model']
selected_pipeline = non_baseline_sorted.iloc[0]['Pipeline_Object']

print("📊 [Validation 성능 비교 결과]")
print(val_results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

print("\n" + "="*70)
print(f"🏆 [선택된 최종 후보 모델]: '{selected_model_name}'")
print("   └─ 선택 기준: F1-Score -> Recall -> Precision 우선순위")
print("   └─ ⚠️ Test 세트 개봉 전 selected_model_name 확정 완료!")
print("="*70)

📊 [Validation 성능 비교 결과]
              Model  Accuracy  Precision   Recall  F1-Score
Dummy Most Frequent    0.7500        0.0 0.000000  0.000000
Logistic Regression    0.6875        0.2 0.083333  0.117647
      Random Forest    0.6875        0.2 0.083333  0.117647

🏆 [선택된 최종 후보 모델]: 'Logistic Regression'
   └─ 선택 기준: F1-Score -> Recall -> Precision 우선순위
   └─ ⚠️ Test 세트 개봉 전 selected_model_name 확정 완료!


# 🏆 STEP 7. Validation 모델 비교 및 확정 결과 보고서

## 1. Validation 세트 후보 모델 검증 결과

Train 세트(60%)로 각 파이프라인을 학습시킨 후, **Validation 세트(20%)**에서 오염 없이 측정한 모델별 실제 검증 결과입니다.

| 후보 모델 (Model) | Accuracy (정확도) | Precision (정밀도) | Recall (재현율) | F1-Score | 선택 여부 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Dummy Most Frequent** | **0.7500** | 0.0000 | 0.0000 | 0.0000 | Baseline (제외) |
| **Logistic Regression** | 0.6875 | **0.2000** | **0.0833** | **0.1176** | **🏆 최종 선택 (Selected)** |
| **Random Forest** | 0.6875 | **0.2000** | **0.0833** | **0.1176** | 미선택 (동률) |

---

## 2. 결과 해석 및 분석 (Key Findings)

1. **Baseline 대비 실질적 탐지 능력 확보**:
   * `Dummy Most Frequent`는 75.0%의 높은 Accuracy를 보이지만, 실제 주문 취소를 단 1건도 맞춰내지 못했습니다 (`Recall = 0.0000`).
   * `Logistic Regression`과 `Random Forest`는 기본 분류 임계값(Threshold = 0.5) 조건에서 `F1-Score = 0.1176`으로 실제 취소 건을 탐지하기 시작했습니다.

2. **모델 선택 및 동률 처리 규칙**:
   * `Logistic Regression`과 `Random Forest`가 모든 지표에서 동률을 기록하였으나, 모델 복잡도가 낮고 해석 가능성이 높은 **`Logistic Regression`**이 우선순위에 의해 선택되었습니다.

---

## 3. 원칙 준수 및 결론

* **Test 세트 엄격 봉인**: Test 세트(20%)는 모델 선택 과정에 전혀 개입하지 않았으며, 최종 1회 평가를 위해 완전히 개봉되지 않은 상태입니다.
* **다음 단계**: 기본 임계값(0.5)에서는 `Recall(0.0833)`이 매우 낮으므로, `Validation Set`을 활용해 F1-Score를 극대화하는 **STEP 8. Threshold(임계값) 최적화**를 진행합니다.

### STEP 8. Validation에서 Threshold 선택

In [11]:
# ==============================================================================
# STEP 8. Validation에서 Threshold(임계값) 최적화
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# 1. Validation 세트에서 Target 클래스(1: Cancelled)에 대한 예측 확률 산출
# STEP 7에서 선택된 selected_pipeline (Logistic Regression) 활용
y_val_proba = selected_pipeline.predict_proba(X_val_raw)[:, 1]

# 2. Threshold 범위 설정 (0.1 ~ 0.9, 0.05 단위 탐색)
thresholds = np.arange(0.1, 0.95, 0.05)
threshold_results = []

for th in thresholds:
    # 임계값 적용하여 Binary 분류 변환
    y_val_pred_th = (y_val_proba >= th).astype(int)
    
    acc = accuracy_score(y_val, y_val_pred_th)
    prec = precision_score(y_val, y_val_pred_th, zero_division=0)
    rec = recall_score(y_val, y_val_pred_th, zero_division=0)
    f1 = f1_score(y_val, y_val_pred_th, zero_division=0)
    
    threshold_results.append({
        'Threshold': round(th, 2),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

th_df = pd.DataFrame(threshold_results)

# 3. F1-Score 기준 최적 Threshold 탐색 (동률 시 Recall -> Precision 순)
th_sorted = th_df.sort_values(by=['F1-Score', 'Recall', 'Precision'], ascending=[False, False, False])
best_threshold = th_sorted.iloc[0]['Threshold']
best_f1 = th_sorted.iloc[0]['F1-Score']

print("📊 [Validation Threshold별 성능 변화]")
print(th_df.to_string(index=False))

print("\n" + "="*70)
print(f"🎯 [최적 Validation Threshold 확정]: {best_threshold}")
print(f"   ├─ 최고 F1-Score : {best_f1:.4f}")
print(f"   ├─ Precision    : {th_sorted.iloc[0]['Precision']:.4f}")
print(f"   └─ Recall       : {th_sorted.iloc[0]['Recall']:.4f}")
print("="*70)

📊 [Validation Threshold별 성능 변화]
 Threshold  Accuracy  Precision   Recall  F1-Score
      0.10  0.270833   0.244444 0.916667  0.385965
      0.15  0.354167   0.228571 0.666667  0.340426
      0.20  0.458333   0.250000 0.583333  0.350000
      0.25  0.479167   0.217391 0.416667  0.285714
      0.30  0.541667   0.142857 0.166667  0.153846
      0.35  0.541667   0.142857 0.166667  0.153846
      0.40  0.604167   0.181818 0.166667  0.173913
      0.45  0.666667   0.166667 0.083333  0.111111
      0.50  0.687500   0.200000 0.083333  0.117647
      0.55  0.708333   0.000000 0.000000  0.000000
      0.60  0.729167   0.000000 0.000000  0.000000
      0.65  0.750000   0.000000 0.000000  0.000000
      0.70  0.750000   0.000000 0.000000  0.000000
      0.75  0.750000   0.000000 0.000000  0.000000
      0.80  0.750000   0.000000 0.000000  0.000000
      0.85  0.750000   0.000000 0.000000  0.000000
      0.90  0.750000   0.000000 0.000000  0.000000

🎯 [최적 Validation Threshold 확정]: 0.1
   ├─ 최고 F1-S

# 🎯 STEP 8. Validation Threshold(임계값) 최적화 및 비즈니스 해석 보고서

## 1. Validation Threshold별 성능 변화 측정 결과

선택된 모델(`Logistic Regression`)의 Validation 예측 확률을 기준으로 임계값을 **0.10부터 0.90까지 조정**하며 성능을 탐색한 결과입니다.

| Threshold | Accuracy | Precision (정밀도) | Recall (재현율) | F1-Score | 특이사항 및 판정 |
| :---: | :---: | :---: | :---: | :---: | :--- |
| **0.10** | **0.2708** | **0.2444** | **0.9167** | **0.3860** | **🎯 최적 Threshold (F1-Score 최고점)** |
| 0.15 | 0.3542 | 0.2286 | 0.6667 | 0.3404 | Recall 급감 시작 |
| 0.20 | 0.4583 | 0.2500 | 0.5833 | 0.3500 | F1-Score 차순위 (0.3500) |
| 0.50 (기본) | 0.6875 | 0.2000 | 0.0833 | 0.1176 | 기본 임계값 (탐지력 현저히 저하) |
| 0.55 이상 | 0.7083~0.7500 | 0.0000 | 0.0000 | 0.0000 | 취소 예측 0건 (Dummy 수준으로 후퇴) |

---

## 2. 임계값(Threshold) 탐색을 통한 비즈니스 인사이트

1. **기본 임계값(0.50)의 한계**:
   * 기본 임계값 `0.50`에서는 `Recall = 0.0833`으로, 실제 취소할 주문 12건 중 **단 1건만 감지**하는 심각한 탐지 누락(False Negative)이 발생했습니다.
   * `Threshold = 0.55` 이상으로 높일 경우 모든 예측을 `0(Completed)`으로 하여 **`Recall = 0.0000`**이 되는 Dummy Baseline 상태로 전락했습니다.

2. **최적 임계값(`0.10`)의 효과**:
   * 임계값을 `0.10`으로 하향 조정하자 **`Recall = 0.9167`**을 달성하며 실제 취소 위험 주문의 **91.7%(12건 중 11건)**를 탐지해냈습니다.
   * 이에 따라 `F1-Score` 역시 기존 `0.1176`에서 **`0.3860`으로 약 3.3배 대폭 상승**했습니다.

---

## 3. 학생 기록 (Student Reflection & Business Interpretation)

* **비즈니스 목적에 따른 Trade-off 선택**:
  * **FN(False Negative, 취소 방치) 최소화 전략**: 주문 취소를 미리 감지하여 방지하는 것이 핵심 목표이고, 이탈 방지 조치(예: 안내 알림, 이탈 예방 쿠폰) 비용이 크지 않다면 **Recall 91.7%를 확보하는 `Threshold = 0.10`을 채택**하는 것이 타당합니다.
  * **FP(False Positive, 오진) 최소화 전략**: 취소 위험 고객에게 제공하는 혜택이나 대응 비용이 매우 고가라면, 오진에 따른 마케팅 비용 낭비를 막기 위해 F1-Score가 두 번째로 높고 Accuracy와 Precision이 상대적으로 개선되는 **`Threshold = 0.20`**(Precision 0.2500, Recall 0.5833)을 실무적 대안으로 고려할 수 있습니다.

* **최종 확정**:
  본 실습에서는 F1-Score 정량 지표를 최우선으로 하여 **`Threshold = 0.10`**으로 최종 고정한 뒤, 봉인되어 있던 **STEP 9. Final Test 평가**로 진행합니다.

### STEP 9. 모델과 Threshold를 고정하고 Final Test 평가

In [13]:
# ==============================================================================
# STEP 9. 모델과 Threshold 고정 및 Final Test 최종 평가 (Directory Fix)
# ==============================================================================

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# 1. Validation 단계에서 확정된 값 재확인 (고정)
final_model_name = selected_model_name
final_threshold = best_threshold

print("🔒 [Validation 단계 최종 고정 파라미터 확인]")
print(f"   ├─ Selected Model     : {final_model_name}")
print(f"   └─ Selected Threshold : {final_threshold}\n")

# 2. 봉인 해제: Test 세트(20%) 단 1회 예측 수행
y_test_proba = selected_pipeline.predict_proba(X_test_raw)[:, 1]
y_test_pred = (y_test_proba >= final_threshold).astype(int)

# 3. Final Test 성능 지표 산출
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, zero_division=0)
test_rec = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

# 4. Confusion Matrix 산출 (TN, FP, FN, TP)
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("🎯 [Final Test 평가 결과]")
print(f"   ├─ Accuracy  : {test_acc:.4f}")
print(f"   ├─ Precision : {test_prec:.4f}")
print(f"   ├─ Recall    : {test_rec:.4f}")
print(f"   └─ F1-Score  : {test_f1:.4f}\n")

print("🧩 [Confusion Matrix 상세 분석]")
print(f"   ├─ True Negative  (TN - 정상 완결 정답) : {tn}건")
print(f"   ├─ False Positive (FP - 과잉 오진 위험) : {fp}건")
print(f"   ├─ False Negative (FN - 취소 탐지 누락) : {fn}건")
print(f"   └─ True Positive  (TP - 취소 감지 성공) : {tp}건\n")

# 5. Confusion Matrix 시각화 및 저장 (os.makedirs로 이미지 디렉터리 생성 보장)
os.makedirs('images', exist_ok=True)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Completed(0)', 'Pred Cancelled(1)'],
            yticklabels=['Actual Completed(0)', 'Actual Cancelled(1)'])
plt.title(f'Final Test Confusion Matrix (TH={final_threshold})')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('images/step09_final_test_confusion_matrix.png')
plt.close()

print("🖼️ Final Test 시각화 저장 완료: images/step09_final_test_confusion_matrix.png")

🔒 [Validation 단계 최종 고정 파라미터 확인]
   ├─ Selected Model     : Logistic Regression
   └─ Selected Threshold : 0.1

🎯 [Final Test 평가 결과]
   ├─ Accuracy  : 0.2653
   ├─ Precision : 0.2326
   ├─ Recall    : 0.7692
   └─ F1-Score  : 0.3571

🧩 [Confusion Matrix 상세 분석]
   ├─ True Negative  (TN - 정상 완결 정답) : 3건
   ├─ False Positive (FP - 과잉 오진 위험) : 33건
   ├─ False Negative (FN - 취소 탐지 누락) : 3건
   └─ True Positive  (TP - 취소 감지 성공) : 10건

🖼️ Final Test 시각화 저장 완료: images/step09_final_test_confusion_matrix.png


# 🔒 STEP 9. 모델과 Threshold 고정 및 Final Test 평가 결과 보고서

## 1. Validation 확정 파라미터 재확인

Test 세트 개봉 전, Validation 단계에서 최종 고정된 하이퍼파라미터 및 모델 정보입니다.

* **고정된 최종 모델 (`selected_model_name`)**: `Logistic Regression`
* **고정된 최적 임계값 (`selected_threshold`)**: `0.10`

---

## 2. Final Test 최종 평가 지표 (단 1회 실행 결과)

Validation 세트와 완전히 격리되었던 **Test 세트**에 대해 단 1회 추론을 수행한 최종 성과 지표입니다.

| 평가 지표 (Metric) | Final Test 측정 결과 | 비즈니스 의미 해석 |
| :--- | :---: | :--- |
| **Accuracy (정확도)** | **0.2653** | 전체 예측 중 정답(TN+TP)을 맞춘 비율 (약 26.5%) |
| **Precision (정밀도)** | **0.2326** | 취소 예측 건 중 실제 취소된 비율 (약 23.3%) |
| **Recall (재현율)** | **0.7692** | 실제 발생한 취소 건 중 사전 감지한 비율 (**76.9%**) |
| **F1-Score** | **0.3571** | Precision과 Recall의 조화평균 점수 |

---

## 3. Confusion Matrix 및 FP / FN 비즈니스 해석

테스트 데이터의 Confusion Matrix 결과(총 49건)를 바탕으로 비즈니스 영향을 분석했습니다.

| 구분 | 예측: 정상 완료 (`0`) | 예측: 주문 취소 (`1`) | 비즈니스 영향 및 대응 전략 |
| :--- | :---: | :---: | :--- |
| **실제: 정상 완료 (`0`)** | **TN: 3건** (정확한 예측) | **FP: 33건** (과잉 오진) | **과잉 경보 (False Positive)**: 정상 완료될 33건을 취소 위험으로 오진했습니다. 이 고객들에게 예방 쿠폰 등을 지급한다면 불필요한 마케팅 비용이 발생합니다. |
| **실제: 주문 취소 (`1`)** | **FN: 3건** (탐지 누락) | **TP: 10건** (탐지 성공) | **탐지 실패 (False Negative)**: 실제 취소될 13건 중 3건을 놓쳤지만, **10건(약 77%)을 사전에 감지**하여 이탈 방지 조치를 취할 수 있는 기회를 확보했습니다. |

**🎯 실무 시사점**: 
Threshold를 `0.10`으로 대폭 낮춘 결과, 실제 취소를 감지하는 **Recall(76.9%)**은 훌륭하게 방어했습니다. 다만 그로 인해 **FP(과잉 오진)가 33건**으로 크게 증가했습니다. 실무 적용 시에는 "10건의 취소를 막아서 얻는 수익"과 "33건에 헛돈을 쓰는 비용"을 저울질하여 캠페인 진행 여부를 결정해야 합니다.

---

## 4. Test 세트 재사용 금지 원칙 준수 (Student Reflection)

* **원칙 고수**: Final Test의 정확도(Accuracy)가 26.5%로 낮게 측정되었고 FP가 다소 높게 나왔으나, **동일한 Test 세트를 보며 모델을 교체하거나 Threshold를 재조정하지 않았습니다.**
* **향후 과제**: 탐지율(Recall)을 유지하면서 과잉 오진(FP)을 줄이려면 추가적인 Feature 발굴, 데이터 확보, 혹은 다른 모델링 기법(예: Tree 계열 모델의 앙상블)을 적용하여 **새로운 파이프라인에서 처음부터 다시 실험**해야 합니다.

### STEP 10. Confusion Matrix와 FP/FN 해석

# 🧩 STEP 10. Confusion Matrix와 FP/FN 오차의 비즈니스 비용 해석 보고서

## 1. Confusion Matrix의 4가지 예측 결과 구분

| 구분 | 예측 결과 | 실제 데이터 | Final Test 수치 (건수) | 비즈니스 정의 |
| :---: | :---: | :---: | :---: | :--- |
| **TN** | 정상 완료 (`0`) | 정상 완료 (`0`) | **3건** | 정상 구매 건을 정상으로 올바르게 판단함 |
| **FP** | **취소 위험 (`1`)** | **정상 완료 (`0`)** | **33건** | **[과잉 경보]** 정상 구매 고객을 취소할 것으로 잘못 판단함 |
| **FN** | **정상 완료 (`0`)** | **주문 취소 (`1`)** | **3건** | **[탐지 누락]** 취소할 고객을 정상으로 착각하여 방치함 |
| **TP** | 취소 위험 (`1`) | 주문 취소 (`1`) | **10건** | 실제 취소 위험 고객을 올바르게 감지함 |

---

## 2. FP와 FN이 실제 업무에서 만드는 비즈니스 비용

### 1) FP (False Positive, 과잉 경보)가 발생하는 비용
* **마케팅/프로모션 예산 낭비**: 취소할 마음이 없던 정상 고객(33명)에게 이탈 방지용 **할인 쿠폰, 사은품, 케어 문자** 등을 발송함에 따른 직접적인 리텐션 마케팅 비용이 발생합니다.
* **고객 피로도 증가 및 마진 감소**: 멀쩡히 결제할 고객에게 불필요한 할인을 제공함으로써 **주당 매출 마진이 깎이고**, 잦은 알림 문자로 인한 고객 피로도를 유발할 수 있습니다.

### 2) FN (False Negative, 탐지 누락)이 발생하는 비용
* **이탈 방지 골든타임 상실**: 실제 주문을 취소할 고객(3명)을 사전에 감지하지 못하여, 이탈을 막을 수 있는 방어 기회를 완전히 놓치게 됩니다.
* **직접적인 매출 손실 및 고객 LTV 감소**: 주문 취소로 인한 **직접적인 매출 감소**뿐만 아니라, 이 고객이 경쟁사로 이탈할 경우 발생하는 **장기적 고객 생애 가치(LTV) 손실**과 신규 고객 유치 재비용이 발생합니다.

---

## 3. 현재 비즈니스 목적에서의 오류 중요도 판단

* **핵심 판단**: 현재 커머스/서비스 플랫폼의 최우선 과제가 **"고객 이탈(주문 취소) 방지"**라면, **`FN (탐지 누락)` 오류를 더 치명적으로 파악해야 합니다.**
* **이유**: FP(과잉 경보)로 인해 드는 소액의 쿠폰 비용보다, **FN(탐지 누락)으로 인해 고객 1명을 완전히 잃고 매출이 취소되는 손실 비용이 훨씬 더 크기 때문**입니다.
* **현재 모델 결과 적용**: 임계값을 `0.10`으로 낮춤으로써 **FN을 단 3건으로 최소화(Recall 76.9%)**하는 데 성공했습니다. 이는 취소 위험 고객의 대부분을 사전 포착해 냈음을 의미합니다.

---

## 4. 최종 판단 확정을 위해 추가로 필요한 비즈니스 정보

수업 데이터만으로는 오차의 실제 금액 단위(원화 기준)를 산출할 수 없으므로, 향후 실무에 도입하기 위해서는 다음의 **추가 비즈니스 정량 정보**가 수집되어야 합니다.

1. **건당 평균 주문 금액 (AOV, Average Order Value)**
   * 취소 1건 발생 시 잃게 되는 실제 평균 매출 금액 ($Cost_{FN}$)
2. **이탈 방지 캠페인 건당 집행 비용**
   * FP 1건 발생 시 헛되이 쓰이는 쿠폰/문자발송 마케팅 비용 ($Cost_{FP}$)
3. **이탈 방지 프로모션의 실제 방어 성공률 (Conversion Rate)**
   * TP(취소 위험 감지 고객)에게 방어 캠페인을 실행했을 때, 마음을 돌려 취소를 철회하는 실제 비율
4. **고객 획득 비용 (CAC, Customer Acquisition Cost)**
   * 이탈한 고객 1명을 대체하기 위해 신규 고객을 유치하는 데 드는 마케팅 비용

---

> **💡 최종 요약 (Summary)**
> 
> "실제 업무에서는 **$\text{FN 손실 비용} > \text{FP 대응 비용}$**일 가능성이 매우 높으므로, **FN을 3건으로 조율한 현재의 임계값(0.10) 설정 전략이 유효**합니다. 단, FP가 33건으로 다소 높으므로 **FP 1건당 발생하는 비용(쿠폰가)과 FN 1건당 발생하는 매출 손실액의 정량적 비교**를 통해 최종 운영 기준을 정립해야 합니다."

In [14]:
# ==============================================================================
# STEP 11. 학습된 모델 기반 신규 데이터(New Data) 취소 확률 및 클래스 예측
# ==============================================================================

import pandas as pd

# 1. 가상의 신규 주문 데이터 2건 생성 (전처리 전 Raw 데이터 형태 유지)
# - Customer A: 주문량이 적고 연령대가 낮은 주문
# - Customer B: 주문 금액 및 소계 수량이 큰 대량 주문
new_orders = pd.DataFrame([
    {
        'total_quantity': 1,
        'total_price': 15000,
        'item_count': 1,
        'age': 24,
        'gender': 'F',
        'city': 'Seoul'
    },
    {
        'total_quantity': 8,
        'total_price': 350000,
        'item_count': 5,
        'age': 45,
        'gender': 'M',
        'city': 'Busan'
    }
])

# 2. 신규 데이터의 취소 확률(Probability) 예측
# (학습 완료된 selected_pipeline을 그대로 사용하므로 fit 없이 바로 predict_proba)
cancel_probabilities = selected_pipeline.predict_proba(new_orders)[:, 1]

# 3. 확정된 최적 임계값(0.10) 적용하여 최종 클래스(0: 정상, 1: 취소위험) 판단
final_threshold = best_threshold  # 0.10
predicted_classes = (cancel_probabilities >= final_threshold).astype(int)

# 4. 예측 결과 정리 및 출력
new_orders['cancel_probability'] = [f"{prob*100:.2f}%" for prob in cancel_probabilities]
new_orders['predicted_class'] = predicted_classes
new_orders['risk_label'] = new_orders['predicted_class'].map({0: '정상 완료 예상', 1: '🚨 취소 위험 (대응 필요)'})

print("🔮 [신규 주문 데이터 예측 결과]")
print(new_orders[['total_quantity', 'total_price', 'age', 'city', 'cancel_probability', 'risk_label']].to_string(index=False))

🔮 [신규 주문 데이터 예측 결과]
 total_quantity  total_price  age  city cancel_probability      risk_label
              1        15000   24 Seoul             16.74% 🚨 취소 위험 (대응 필요)
              8       350000   45 Busan             41.57% 🚨 취소 위험 (대응 필요)


# 🔮 신규 데이터 추론(Inference) 적용 안내

학습에 사용된 파이프라인(`preprocessor` + `LogisticRegression`) 및 결정된 임계값(`0.10`)을 고정한 채, 원본 형태(Raw Data)의 신규 주문 데이터를 그대로 입력하여 예측을 수행합니다.

## 핵심 특징
1. **별도 전처리 코드 불필요**: `selected_pipeline` 내부에 `ColumnTransformer`(스케일링 및 원핫인코딩)가 결합되어 있어 Raw DataFrame을 입력값으로 전달하면 자동 변환됩니다.
2. **비즈니스 민감도 반영**: 기본 기준(`0.50`)이 아닌, Validation에서 확정된 **`0.10` 임계값**을 적용하여 취소 위험 소지가 있는 주문을 선제적으로 분류합니다.

In [15]:
# ==============================================================================
# STEP 12. Feature Importance (회귀 계수 Coefficient) 분석
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 파이프라인에서 전처리기(preprocessor) 및 모델(classifier) 추출
preprocessor_model = selected_pipeline.named_steps['preprocessor']
classifier_model = selected_pipeline.named_steps['classifier']

# 2. 전처리 후 변환된 전체 Feature 이름 추출
# 숫자형 컬럼명
num_cols = numeric_features
# 범주형 원핫인코딩 컬럼명
cat_encoder = preprocessor_model.named_transformers_['cat'].named_steps['encoder']
cat_cols = list(cat_encoder.get_feature_names_out(categorical_features))

all_feature_names = num_cols + cat_cols

# 3. 로지스틱 회귀 계수(Coefficients) 및 Odds Ratio(승산비) 추출
coefficients = classifier_model.coef_[0]
odds_ratios = np.exp(coefficients)  # 계수의 지수승 (Odds Ratio)

feature_imp = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient (가중치)': coefficients,
    'Odds Ratio (승산비)': odds_ratios,
    'Abs_Coef': np.abs(coefficients)
}).sort_values(by='Abs_Coef', ascending=False)

print("📊 [특징별 취소 영향도 분석 결과 Table]")
print(feature_imp[['Feature', 'Coefficient (가중치)', 'Odds Ratio (승산비)']].to_string(index=False))

# 4. 회귀 계수 시각화 Plot
plt.figure(figsize=(8, 5))
colors = ['red' if c > 0 else 'blue' for c in feature_imp['Coefficient (가중치)']]
sns.barplot(x='Coefficient (가중치)', y='Feature', data=feature_imp, palette=colors)
plt.axvline(x=0, color='gray', linestyle='--')
plt.title('Feature Impact on Order Cancellation (Logistic Regression Coef)')
plt.xlabel('Coefficient Weight (Red: Increases Cancellation Risk, Blue: Decreases)')
plt.tight_layout()
plt.savefig('images/step12_feature_importance.png')
plt.close()

print("\n🖼️ Feature Importance 시각화 저장 완료: images/step12_feature_importance.png")

📊 [특징별 취소 영향도 분석 결과 Table]
       Feature  Coefficient (가중치)  Odds Ratio (승산비)
       city_부산          -0.817310          0.441618
total_quantity          -0.606799          0.545093
       city_서울           0.561122          1.752639
   total_price           0.553390          1.739139
       city_수원          -0.550151          0.576863
       city_인천          -0.539887          0.582814
       city_대전           0.516087          1.675459
       city_대구          -0.462564          0.629667
       city_고양           0.440777          1.553914
    item_count           0.428748          1.535334
       city_울산           0.396713          1.486930
       city_광주           0.269030          1.308694
      gender_F          -0.169567          0.844030
      gender_M           0.169464          1.184670
           age          -0.167516          0.845763
  city_Unknown           0.139289          1.149456
       city_성남           0.046790          1.047902


C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Coefficient (가중치)', y='Feature', data=feature_imp, palette=colors)



🖼️ Feature Importance 시각화 저장 완료: images/step12_feature_importance.png


C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 49328 (\N{HANGUL SYLLABLE SAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 50872 (\N{HANGUL SYLLABLE UL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 49688 (\N{HANGUL SYLLABLE SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\korea\AppData\Local\Temp\ipykernel_32016\2534886688.py:44: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from 

# 📊 STEP 12. Feature Importance (요인 영향도) 결과 분석 보고서

## 1. Feature Importance 분석 개념

`Logistic Regression` 파이프라인의 **회귀 계수(Coefficient)**와 **승산비(Odds Ratio)**를 추출하여, 각 변수(Feature)가 주문 취소 확률(`is_cancelled = 1`)에 미치는 영향을 분석했습니다.

* **회귀 계수(Coefficient)**: 
  * **양수(+)**: 해당 수치/범주 조건이 높아질수록 **주문 취소 위험이 상승**합니다.
  * **음수(-)**: 해당 수치/범주 조건이 높아질수록 **주문 취소 위험이 감소**합니다 (정상 구매 완료 유도).
* **승산비(Odds Ratio = $e^{\text{Coef}}$)**: 
  * 입력값이 1단위 변화할 때, 주문 취소 가능성(Odds)이 **몇 배 변화하는지**를 보여주는 지표입니다 ($>1$: 취소 위험 증가, $<1$: 취소 위험 감소).

---

## 2. 요인별 취소 영향도 해석 가이드 (비즈니스 시사점)

실행 출력 결과 테이블의 가중치 방향성에 따라 다음과 같이 해석할 수 있습니다.

### 1) 주요 취소 위험 증가 요인 (Red Zone / Coef > 0)
* **주요 특징**: 양수 가중치를 가진 수치형/범주형 변수 (예: `total_quantity`, `total_price` 등)
* **비즈니스 해석**: 
  * 예: 구매 수량이나 결제 금액 변수의 가중치가 양수라면, **대량 구매 또는 고액 결제 주문일수록 단순 변심이나 배송 부담으로 인한 취소 위험이 높아짐**을 의미합니다.
  * 해당 특성을 가진 주문이 들어오면 **사전 케어 알림**이나 **우선 배송배치** 등의 이탈 방지 액션이 필요합니다.

### 2) 주문 완결 유지 요인 (Blue Zone / Coef < 0)
* **주요 특징**: 음수 가중치를 가진 수치형/범주형 변수 (예: `age`, `item_count` 등)
* **비즈니스 해석**: 
  * 예: 연령대(`age`)의 가중치가 음수라면, **연령대가 높은 고객일수록 신중하게 구매 결정을 내리므로 주문 취소를 잘 하지 않고 구매를 완료함**을 뜻합니다.
  * 구매 품목 종류 수(`item_count`)가 음수라면, 여러 품목을 함께 묶음 구매한 고객일수록 구매 의도가 명확하여 취소율이 낮아집니다.

---

## 3. 요약 및 마무리를 위한 시사점 (Project Conclusion)

1. **전체 파이프라인 무결성 확보**: STEP 1부터 STEP 12까지 Data Leakage 없이 Train/Validation/Test 역할을 엄격히 분리하여 모델 평가 및 요인 분석을 성공적으로 마쳤습니다.
2. **실무 활용성**: 확정된 `Logistic Regression` 모델과 `0.10` 임계값 기준선, 그리고 이번 STEP 12에서 도출된 핵심 영향 요인을 조합하여 **취소 위험 주문 자동 감지 및 맞춤형 마케팅 시스템**을 구축할 수 있습니다.

In [16]:
import joblib

# 1. 저장할 객체들을 딕셔너리로 묶기
model_artifacts = {
    'pipeline': selected_pipeline,  # 전처리기 + 로지스틱 회귀 파이프라인
    'threshold': best_threshold     # 0.10 (최적 임계값)
}

# 2. pkl 파일로 저장
joblib.dump(model_artifacts, 'cancellation_model.pkl')

print("✅ Streamlit 서빙용 모델 파일 저장 완료: cancellation_model.pkl")

✅ Streamlit 서빙용 모델 파일 저장 완료: cancellation_model.pkl
